In [1]:
import os
import pandas as pd
import numpy as np


In [2]:
import os

all_columns = set()
folder_path = "processed_training_Ay"

for f in os.listdir(folder_path):
    if f.endswith(".psv"):
        # Open the file and read only the first line (the header)
        with open(os.path.join(folder_path, f), "r") as file:
            header = file.readline().strip()
            if header:
                # Split by the pipe delimiter and update the set
                all_columns.update(header.split("|"))

FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")

print("Final feature count:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Final feature count: 32
['cat__Gender_0.0', 'cat__Gender_1.0', 'cat__Unit1_0.0', 'cat__Unit1_1.0', 'cat__Unit2_0.0', 'cat__Unit2_1.0', 'num__Age', 'num__BUN', 'num__BaseExcess', 'num__Chloride', 'num__Creatinine', 'num__DBP', 'num__FiO2', 'num__Glucose', 'num__HCO3', 'num__HR', 'num__Hct', 'num__Hgb', 'num__HospAdmTime', 'num__ICULOS', 'num__MAP', 'num__Magnesium', 'num__O2Sat', 'num__PaCO2', 'num__Platelets', 'num__Potassium', 'num__Resp', 'num__SBP', 'num__SaO2', 'num__Temp', 'num__WBC', 'num__pH']


In [3]:
def load_patient_sequences(folder, feature_columns, early_hours=6):
    X, y = [], []

    for f in os.listdir(folder):
        df = pd.read_csv(os.path.join(folder, f), sep="|")

        # 🔑 FORCE SAME FEATURE SPACE
        df = df.reindex(columns=feature_columns + ["SepsisLabel"])
        df[feature_columns] = df[feature_columns].fillna(0.0)

        features = df[feature_columns].values
        labels = df["SepsisLabel"].values

        if labels.max() == 1:
            t_sepsis = np.where(labels == 1)[0][0]
            cutoff = t_sepsis - early_hours
            if cutoff <= 0:
                continue
            seq = features[:cutoff]
            label = 1
        else:
            seq = features
            label = 0

        X.append(seq)
        y.append(label)

    return X, np.array(y)

In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

I0000 00:00:1780645154.228553   87251 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780645154.273612   87251 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780645155.451333   87251 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
def pad_data(X, max_len=None):
    if max_len is None:
        max_len = max(len(x) for x in X)

    X_pad = pad_sequences(
        X,
        maxlen=max_len,
        dtype="float32",
        padding="pre",
        truncating="pre"
    )

    return X_pad, max_len

In [6]:
# TRAIN (Set A)
X_train_raw, y_train = load_patient_sequences("processed_training", FEATURE_COLUMNS)
X_train, max_len = pad_data(X_train_raw)

# VALIDATION / TEST (Set B)
X_val_raw, y_val = load_patient_sequences("processed_training_setB", FEATURE_COLUMNS)
X_val, _ = pad_data(X_val_raw, max_len=max_len)

In [7]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

TIMESTEPS = X_train.shape[1]
N_FEATURES = X_train.shape[2]

print("TIMESTEPS:", TIMESTEPS)
print("N_FEATURES:", N_FEATURES)

assert N_FEATURES == 32

X_train shape: (19900, 336, 32)
X_val shape: (19658, 336, 32)
TIMESTEPS: 336
N_FEATURES: 32


In [8]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=y_train
)

class_weight = {0: class_weights[0], 1: class_weights[1]}
print(class_weight)


{0: np.float64(0.536503828318775), 1: np.float64(7.348596750369277)}


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

def build_lstm(units=64, dropout=0.3):
    model = Sequential([
        Input(shape=(TIMESTEPS, N_FEATURES)),
        LSTM(units, return_sequences=False),
        Dropout(dropout),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


In [10]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    "units": [32, 64],
    "dropout": [0.3, 0.5]
}

best_auc = 0.0
best_model = None

for params in ParameterGrid(param_grid):
    print("Training with:", params)

    model = build_lstm(
        units=params["units"],
        dropout=params["dropout"]
    )

    model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        class_weight=class_weight,
        validation_data=(X_val, y_val),
        verbose=1
    )

    val_auc = model.evaluate(X_val, y_val, verbose=0)[1]

    if val_auc > best_auc:
        best_auc = val_auc
        best_model = model

Training with: {'dropout': 0.3, 'units': 32}


E0000 00:00:1780645337.359920   87251 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1780645337.387190   87251 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/10


W0000 00:00:1780645339.255039   87251 cpu_allocator_impl.cc:82] Allocation of 855859200 exceeds 10% of free system memory.


622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - auc: 0.6120 - loss: 0.6767

...

Training with: {'dropout': 0.3, 'units': 64}
Epoch 1/10


W0000 00:00:1780645715.416874   87251 cpu_allocator_impl.cc:82] Allocation of 855859200 exceeds 10% of free system memory.


622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - auc: 0.6676 - loss: 0.6353

W0000 00:00:1780645755.179317   87251 cpu_allocator_impl.cc:82] Allocation of 845451264 exceeds 10% of free system memory.


622/622 ━━━━━━━━━━━━━━━━━━━━ 54s 84ms/step - auc: 0.7396 - loss: 0.5868 - val_auc: 0.6009 - val_loss: 1.5832
Epoch 2/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - auc: 0.8140 - loss: 0.5053 - val_auc: 0.6706 - val_loss: 1.0519
Epoch 3/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - auc: 0.8242 - loss: 0.4953 - val_auc: 0.5529 - val_loss: 1.4534
Epoch 4/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - auc: 0.8455 - loss: 0.4728 - val_auc: 0.5478 - val_loss: 1.5333
Epoch 5/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - auc: 0.8581 - loss: 0.4552 - val_auc: 0.5234 - val_loss: 1.3453
Epoch 6/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - auc: 0.8647 - loss: 0.4441 - val_auc: 0.6113 - val_loss: 1.2537
Epoch 7/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - auc: 0.8699 - loss: 0.4350 - val_auc: 0.5216 - val_loss: 1.7641
Epoch 8/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - auc: 0.8756 - loss: 0.4258 - val_auc: 0.6195 - val_loss: 0.8763
Epoch 9/10
622/622 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/st

In [11]:
from sklearn.metrics import f1_score

y_prob = best_model.predict(X_val).ravel()

thresholds = np.linspace(0.1, 0.9, 50)
best_t, best_f1 = 0, 0

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)


615/615 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step
Best threshold: 0.9
Best F1: 0.13953488372093023


In [12]:
from sklearn.metrics import classification_report, roc_auc_score
y_pred = (y_prob > best_t).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print(classification_report(y_val, y_pred))


ROC-AUC: 0.6785691086011242
              precision    recall  f1-score   support

           0       0.97      0.78      0.86     18858
           1       0.08      0.47      0.14       800

    accuracy                           0.77     19658
   macro avg       0.53      0.62      0.50     19658
weighted avg       0.94      0.77      0.84     19658



In [14]:
# Save as a standard Keras file (.keras format is recommended in newer TF versions)
model.save('sepsis_prediction_model.keras')

In [15]:
loaded_model = tf.keras.models.load_model('sepsis_prediction_model.keras')

# Print a list of all layers and look at the shapes of their weights
for layer in loaded_model.layers:
    weights = layer.get_weights()
    if weights:
        print(f"Layer: {layer.name}")
        print(f" -> Weights shape: {weights[0].shape}")
        print(f" -> Biases shape: {weights[1].shape}")
        print("-" * 30)

Layer: lstm_3
 -> Weights shape: (32, 256)
 -> Biases shape: (64, 256)
------------------------------
Layer: dense_3
 -> Weights shape: (64, 1)
 -> Biases shape: (1,)
------------------------------
